In [1]:
%pip install 'databricks-sdk[notebook]'

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: "'databricks-sdk[notebook]'": Expected package name at the start of dependency specifier
    'databricks-sdk[notebook]'
    ^


In [2]:
import os
from databricks.connect import DatabricksSession

# Initialize Spark locally when needed
if "spark" not in globals():
    spark = DatabricksSession.builder.serverless().getOrCreate()

#NotebookParameters
if "dbutils" in globals():
    #Execute in Databricks
    dbutils.widgets.dropdown(
        "environment", 
        "dev", 
        ["dev", "staging", "prod"]
    )

    dbutils.widgets.text("organization", "databricks")

    environment = dbutils.widgets.get("environment")
    organization = dbutils.widgets.get("organization").strip()
else:
    #Execute in VSCode with Databricks Connect
    environment = os.getenv("LAKEHOUSE_ENV", "dev")
    organization = os.getenv("GITHUB_ORGANIZATION", "databricks").strip()

catalogs = {
    "dev": "opensource_dev",
    "staging": "opensource_staging",
    "prod": "opensource_prod"
}

catalog = catalogs[environment]
schema_name = "github_analytics"

bronze_table = f"{catalog}.{schema_name}.bronze_github_repositories"
silver_staging_table = f"{catalog}.{schema_name}.silver_staging_github_repositories"

print(f"Source Table: {bronze_table}")
print(f"Target Table: {silver_staging_table}")

Source Table: opensource_dev.github_analytics.bronze_github_repositories
Target Table: opensource_dev.github_analytics.silver_staging_github_repositories


c:\Users\raffa\Desktop\Databricks\ProjectOpenSource\open-source-analytics-lakehouse\.venv\Lib\site-packages\databricks\sdk\_widgets\__init__.py:70: UserWarning: 
To use databricks widgets interactively in your notebook, please install databricks sdk using:
	pip install 'databricks-sdk[notebook]'
Falling back to default_value_only implementation for databricks widgets.
  warnings.warn(


In [3]:
from pyspark.sql import functions as F

#Read Bronze records for the specified organization
df_bronze = (
    spark.read.table(bronze_table)
    .filter(F.col("organization") == organization)
)

#Identify the latest snapshot 
latest_snapshot = (
    df_bronze
    .select("snapshot_id", "ingestion_ts")
    .distinct()
    .orderBy(F.col("ingestion_ts").desc())
    .first()
)

if latest_snapshot is None:
    raise ValueError(f"No Bronze snapshots found for organization: {organization}.")

#Read only the latest snapshot records
df = (
    df_bronze
    .filter(F.col("snapshot_id") == latest_snapshot["snapshot_id"])
)

record_count = df.count()

print(f"Organization: {organization}")
print(f"Latest snapshot: {latest_snapshot['snapshot_id']}")
print(f"Records retrieved: {record_count}")

df.select(
    "id",
    "full_name",
    "description",
    "language",
    "snapshot_id",
    "ingestion_ts"
).show(10, truncate=False)

Organization: databricks
Latest snapshot: 6a7b4914-f985-4953-a2e8-d79a3ba182d0
Records retrieved: 253
+----------+---------------------------------+---------------------------------------------------------------------------------------------+----------+------------------------------------+--------------------------+
|id        |full_name                        |description                                                                                  |language  |snapshot_id                         |ingestion_ts              |
+----------+---------------------------------+---------------------------------------------------------------------------------------------+----------+------------------------------------+--------------------------+
|757044536 |databricks/alertmanager          |Prometheus Alertmanager                                                                      |Go        |6a7b4914-f985-4953-a2e8-d79a3ba182d0|2026-11-15 20:50:02.293025|
|710920829 |databricks/api-linter 

In [4]:
# Select repository attributes for Silver Staging
df_silver_staging = (
    df
    .select(
        "id",
        "name",
        "full_name",
        "description",
        "language",
        "archived",
        "default_branch",
        "stargazers_count",
        "forks_count",
        "organization",
        "snapshot_id",
        "ingestion_ts"
    )
)

#Save the latest snapshot records to the Silver staging table
(  
    df_silver_staging
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "True")
    .saveAsTable(silver_staging_table)
)

print(f"Silver Staging table: {silver_staging_table}")
print(f"Records written: {record_count}")

Silver Staging table: opensource_dev.github_analytics.silver_staging_github_repositories
Records written: 253


In [5]:
from pyspark.sql import functions as F

#Read Bronze records for the specified organization
df_bronze = (
    spark.read.table(bronze_table)
    .filter(F.col("organization") == organization)
)

#Identify the latest snapshot 
latest_snapshot = (
    df_bronze
    .select("snapshot_id", "ingestion_ts")
    .distinct()
    .orderBy(F.col("ingestion_ts").desc())
    .first()
)

if latest_snapshot is None:
    raise ValueError(f"No Bronze snapshots found for organization: {organization}.")

#Read only the latest snapshot records
df = (
    df_bronze
    .filter(F.col("snapshot_id") == latest_snapshot["snapshot_id"])
)

record_count = df.count()

print(f"Organization: {organization}")
print(f"Latest snapshot: {latest_snapshot['snapshot_id']}")
print(f"Records retrieved: {record_count}")

df.select(
    "id",
    "full_name",
    "description",
    "language",
    "snapshot_id",
    "ingestion_ts"
).show(10, truncate=False)

Organization: databricks
Latest snapshot: 6a7b4914-f985-4953-a2e8-d79a3ba182d0
Records retrieved: 253
+----------+---------------------------------+---------------------------------------------------------------------------------------------+----------+------------------------------------+--------------------------+
|id        |full_name                        |description                                                                                  |language  |snapshot_id                         |ingestion_ts              |
+----------+---------------------------------+---------------------------------------------------------------------------------------------+----------+------------------------------------+--------------------------+
|757044536 |databricks/alertmanager          |Prometheus Alertmanager                                                                      |Go        |6a7b4914-f985-4953-a2e8-d79a3ba182d0|2026-11-15 20:50:02.293025|
|710920829 |databricks/api-linter 

In [6]:
# Validate Silver Staging
df_check = spark.read.table(silver_staging_table)

print(f"Silver Staging records: {df_check.count()}")

df_check.select(
    "id",
    "full_name",
    "language",
    "snapshot_id",
    "ingestion_ts"
).show(10, truncate=False)


Silver Staging records: 253
+----------+---------------------------------+----------+------------------------------------+--------------------------+
|id        |full_name                        |language  |snapshot_id                         |ingestion_ts              |
+----------+---------------------------------+----------+------------------------------------+--------------------------+
|757044536 |databricks/alertmanager          |Go        |6a7b4914-f985-4953-a2e8-d79a3ba182d0|2026-11-15 20:50:02.293025|
|710920829 |databricks/api-linter            |NULL      |6a7b4914-f985-4953-a2e8-d79a3ba182d0|2026-11-15 20:50:02.293025|
|811440846 |databricks/app-templates         |TypeScript|6a7b4914-f985-4953-a2e8-d79a3ba182d0|2026-11-15 20:50:02.293025|
|1110502360|databricks/appkit                |TypeScript|6a7b4914-f985-4953-a2e8-d79a3ba182d0|2026-11-15 20:50:02.293025|
|1280150304|databricks/appkit-registry       |TypeScript|6a7b4914-f985-4953-a2e8-d79a3ba182d0|2026-11-15 20:50:02.2930

In [7]:
# Verify the latest snapshot loaded into Silver Staging
df_check = spark.read.table(silver_staging_table)

print(f"Silver Staging records: {df_check.count()}")

df_check.groupBy(
    "snapshot_id",
    "ingestion_ts"
).count().show(truncate=False)


Silver Staging records: 253
+------------------------------------+--------------------------+-----+
|snapshot_id                         |ingestion_ts              |count|
+------------------------------------+--------------------------+-----+
|6a7b4914-f985-4953-a2e8-d79a3ba182d0|2026-11-15 20:50:02.293025|253  |
+------------------------------------+--------------------------+-----+

